# Crime Incident Analysis
## Phase 1: Dataset Understanding and Data Preprocessing

**Platform:** Google Colab

The outputs below were computed from the supplied CSV in a Python environment and saved in this Colab-compatible notebook. They were not run in a live Google Colab session.

**To use:** Upload this notebook through Colab → File → Upload notebook. Outputs are already visible. To rerun, run cells in order and upload the original crime CSV when prompted.

**Objective:** Classify attributes, assess data quality, handle missing values appropriately, and demonstrate normalization and equal-width binning. This notebook covers Phase 1.

### 1. Import libraries and load the dataset
The original dataset is kept as a separate copy. If the CSV is absent, Colab asks you to upload it.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

filename = "Crime_Incidents_in_the_Last_30_Days (1).csv"
if not Path(filename).exists():
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))

df = pd.read_csv(filename)
original_df = df.copy()
display(df.head())

,X,Y,CCN,REPORT_DAT,START_DATE,END_DATE,BLOCK,OFFENSE,METHOD,SHIFT,WARD,ANC,DISTRICT,PSA,NEIGHBORHOOD_CLUSTER,BLOCK_GROUP,CENSUS_TRACT,VOTING_PRECINCT,BID,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID,OCTO_RECORD_ID
0,397953.59,140415.23,26137330,2026/09/26 06:54:25+00,2026/09/25 22:36:00+00,2026/09/25 23:34:00+00,3301 - 3399 BLOCK OF GEORGIA AVENUE NW,THEFT/OTHER,OTHERS,MIDNIGHT,1.0,1E,4,409,Cluster 2,003100 2,3100.0,Precinct 38,NaN,397953.59,140415.23,38.931616,-77.023602,1042684332,NaN
1,398099.12,138058.27,26137481,2026/09/26 05:51:18+00,2026/09/26 04:49:00+00,2026/09/26 05:45:00+00,1500 - 1599 BLOCK OF 7TH STREET NW,THEFT/OTHER,OTHERS,MIDNIGHT,2.0,2G,3,307,Cluster 7,004901 2,4901.0,Precinct 18,NaN,398099.12,138058.27,38.910384,-77.021917,1042684333,NaN
2,397525.88,138018.68,26138263,2026/09/27 23:11:57+00,2026/09/25 00:15:00+00,2026/09/25 02:00:00+00,1200 - 1299 BLOCK OF RHODE ISLAND AVENUE NW,THEFT F/AUTO,OTHERS,EVENING,2.0,2F,3,307,Cluster 7,005001 1,5001.0,Precinct 16,NaN,397525.88,138018.68,38.910026,-77.028526,1042684334,NaN
3,398682.30,134716.49,26139697,2026/09/30 15:06:32+00,2026/09/29 12:00:00+00,2026/09/29 20:00:00+00,700 - 899 BLOCK OF 3RD STREET SW,THEFT/OTHER,OTHERS,DAY,6.0,6D,1,103,Cluster 9,010500 2,10500.0,Precinct 128,SOUTHWEST,398682.30,134716.49,38.880281,-77.015187,1042684335,NaN
4,397209.47,139586.90,26142075,2026/10/05 00:01:20+00,2026/10/04 23:37:00+00,2026/10/05 00:00:00+00,2600 - 2698 BLOCK OF 14TH STREET NW,THEFT/OTHER,OTHERS,EVENING,1.0,1A,3,304,Cluster 2,003600 2,3600.0,Precinct 36,NaN,397209.47,139586.90,38.924152,-77.032181,1042684721,NaN


### 2. Dataset overview
Each row is an incident record. Data types describe storage, while the classification below describes meaning.

In [2]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.info()

Rows: 2073
Columns: 25
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2073 entries, 0 to 2072
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   X                     2073 non-null   float64
 1   Y                     2073 non-null   float64
 2   CCN                   2073 non-null   int64  
 3   REPORT_DAT            2073 non-null   object 
 4   START_DATE            2072 non-null   object 
 5   END_DATE              1895 non-null   object 
 6   BLOCK                 2073 non-null   object 
 7   OFFENSE               2073 non-null   object 
 8   METHOD                2073 non-null   object 
 9   SHIFT                 2073 non-null   object 
 10  WARD                  2069 non-null   float64
 11  ANC                   2069 non-null   object 
 12  DISTRICT              2073 non-null   int64  
 13  PSA                   2073 non-null   int64  
 14  NEIGHBORHOOD_CLUSTER  2069 non-null   object 
 15

### 3. Attribute classification
| Attributes | Measurement scale / type | Explanation |
|---|---|---|
| CCN, OBJECTID, OCTO_RECORD_ID | Nominal identifiers | Record identifiers; OCTO_RECORD_ID is entirely missing in this file |
| OFFENSE, METHOD, SHIFT | Nominal categorical | Unordered categories |
| BLOCK, ANC, NEIGHBORHOOD_CLUSTER, BLOCK_GROUP, VOTING_PRECINCT, BID | Nominal categorical | Location names and codes |
| WARD, DISTRICT, PSA, CENSUS_TRACT | Nominal categorical | Numeric-looking location codes |
| REPORT_DAT, START_DATE, END_DATE | Interval, temporal | Differences between timestamps are meaningful |
| X, Y, XBLOCK, YBLOCK, LATITUDE, LONGITUDE | Interval, continuous coordinates | Positions; zero does not indicate absence |

Nominal/ordinal/interval/ratio describe measurement scale. Discrete/continuous describe quantitative variables. An integer identifier is not a quantitative measurement.

The derived recorded interval used later is ratio-scale and treated as continuous.

### 4. Initial quality assessment
Report missing values and exact duplicate rows before making changes.

In [3]:
quality_report = pd.DataFrame({
    "Missing values": df.isna().sum(),
    "Missing percentage": (df.isna().mean() * 100).round(2)
})
display(quality_report)
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


,Missing values,Missing percentage
X,0,0.00
Y,0,0.00
CCN,0,0.00
REPORT_DAT,0,0.00
START_DATE,1,0.05
END_DATE,178,8.59
BLOCK,0,0.00
OFFENSE,0,0.00
METHOD,0,0.00
SHIFT,0,0.00


### 5. Clean text and handle missing categories
- Remove exact duplicate rows if present.
- Trim surrounding spaces and treat blank text as missing.
- Treat numeric location codes as categories.
- Label missing categories “Unknown”; this is a missingness label, not a guessed location. BID is retained with that label.
- Remove the completely empty OCTO_RECORD_ID column.
- Preserve missing timestamps; an invented mean timestamp could distort the incident timeline.

In [4]:
df = df.drop_duplicates().copy()

text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].str.strip().replace("", np.nan)

code_columns = ["WARD", "DISTRICT", "PSA", "CENSUS_TRACT"]
for col in code_columns:
    df[col] = df[col].astype("Int64").astype("string")

category_columns = [
    "BLOCK", "OFFENSE", "METHOD", "SHIFT", "WARD", "ANC",
    "DISTRICT", "PSA", "NEIGHBORHOOD_CLUSTER", "BLOCK_GROUP",
    "CENSUS_TRACT", "VOTING_PRECINCT", "BID"
]
df[category_columns] = df[category_columns].fillna("Unknown")
df = df.drop(columns=["OCTO_RECORD_ID"])

print("Missing categorical values after cleaning:",
      df[category_columns].isna().sum().sum())
print("Columns after removing the empty column:", df.shape[1])
display(df[["WARD", "BID", "OFFENSE", "SHIFT"]].head())

Missing categorical values after cleaning: 0
Columns after removing the empty column: 24


,WARD,BID,OFFENSE,SHIFT
0,1,Unknown,THEFT/OTHER,MIDNIGHT
1,2,Unknown,THEFT/OTHER,MIDNIGHT
2,2,Unknown,THEFT F/AUTO,EVENING
3,6,SOUTHWEST,THEFT/OTHER,DAY
4,1,Unknown,THEFT/OTHER,EVENING


### 6. Convert dates and check consistency
Unparseable dates become NaT. Compare missing counts before and after conversion to identify newly invalid dates. Coordinate bounds detect impossible values, not every possible location error. Timestamps remain in UTC.

In [5]:
date_columns = ["REPORT_DAT", "START_DATE", "END_DATE"]
for col in date_columns:
    missing_before = df[col].isna().sum()
    df[col] = pd.to_datetime(
        df[col], format="%Y/%m/%d %H:%M:%S%z",
        errors="coerce", utc=True
    )
    print(col, "new invalid dates:", df[col].isna().sum() - missing_before)

invalid_time = df["END_DATE"] < df["START_DATE"]
invalid_location = (
    ~df["LATITUDE"].between(-90, 90)
    | ~df["LONGITUDE"].between(-180, 180)
)
print("End time before start:", invalid_time.sum())
print("Coordinates outside valid limits:", invalid_location.sum())
for col in ["METHOD", "SHIFT"]:
    print(col, "categories:", df[col].unique().tolist())

REPORT_DAT new invalid dates: 0
START_DATE new invalid dates: 0
END_DATE new invalid dates: 0
End time before start: 0
Coordinates outside valid limits: 0
METHOD categories: ['OTHERS', 'KNIFE', 'GUN']
SHIFT categories: ['MIDNIGHT', 'EVENING', 'DAY']


### 7. Create a numerical attribute
Recorded_Interval_Hours is the window between the recorded start and end timestamps. It can reflect uncertainty about when an offense occurred; it is not necessarily the actual duration of the crime. Missing endpoints produce missing intervals. Negative intervals are marked missing.

In [6]:
df["Recorded_Interval_Hours"] = (
    df["END_DATE"] - df["START_DATE"]
).dt.total_seconds() / 3600

df.loc[df["Recorded_Interval_Hours"] < 0, "Recorded_Interval_Hours"] = np.nan

display(df[["START_DATE", "END_DATE", "Recorded_Interval_Hours"]].head())
print("Available intervals:", df["Recorded_Interval_Hours"].notna().sum())
print("Missing intervals:", df["Recorded_Interval_Hours"].isna().sum())
print("Longest recorded interval (hours):", df["Recorded_Interval_Hours"].max())

Available intervals: 1895
Missing intervals: 178
Longest recorded interval (hours): 5304.0


,START_DATE,END_DATE,Recorded_Interval_Hours
0,2026-09-25 22:36:00+00:00,2026-09-25 23:34:00+00:00,0.966667
1,2026-09-26 04:49:00+00:00,2026-09-26 05:45:00+00:00,0.933333
2,2026-09-25 00:15:00+00:00,2026-09-25 02:00:00+00:00,1.750000
3,2026-09-29 12:00:00+00:00,2026-09-29 20:00:00+00:00,8.000000
4,2026-10-04 23:37:00+00:00,2026-10-05 00:00:00+00:00,0.383333


### 8. Min–Max normalization
Formula: (value − minimum) / (maximum − minimum). Nonmissing values are scaled to 0–1. The original hours remain available. Long intervals can compress most normalized values near zero, so we will investigate their distribution in Phase 2.

In [7]:
minimum = df["Recorded_Interval_Hours"].min()
maximum = df["Recorded_Interval_Hours"].max()

if maximum > minimum:
    df["Interval_Normalized"] = (
        (df["Recorded_Interval_Hours"] - minimum) / (maximum - minimum)
    )
else:
    df["Interval_Normalized"] = df["Recorded_Interval_Hours"] * 0

display(df[["Recorded_Interval_Hours", "Interval_Normalized"]].head())
print("Normalized minimum:", df["Interval_Normalized"].min())
print("Normalized maximum:", df["Interval_Normalized"].max())

Normalized minimum: 0.0
Normalized maximum: 1.0


,Recorded_Interval_Hours,Interval_Normalized
0,0.966667,0.000182
1,0.933333,0.000176
2,1.750000,0.000330
3,8.000000,0.001508
4,0.383333,0.000072


### 9. Equal-width binning
Divide the observed range into three equal-width intervals. Short/Medium/Long are relative labels for recorded time windows, not severity levels. The bin boundaries are shown so the labels can be interpreted correctly.

In [8]:
df["Interval_Bin"], boundaries = pd.cut(
    df["Recorded_Interval_Hours"], bins=3,
    labels=["Short interval", "Medium interval", "Long interval"],
    include_lowest=True, retbins=True
)

print("Bin boundaries (hours):", boundaries)
print("The slightly negative first boundary is a pandas inclusion adjustment.")
print(df["Interval_Bin"].value_counts(sort=False))
print("Unassigned due to missing interval:", df["Interval_Bin"].isna().sum())

Bin boundaries (hours): [  -5.304 1768.    3536.    5304.   ]
The slightly negative first boundary is a pandas inclusion adjustment.
Interval_Bin
Short interval     1892
Medium interval       2
Long interval         1
Name: count, dtype: int64
Unassigned due to missing interval: 178


### 10. Final quality check
Missing values in dates and derived columns are intentional. The processed dataset has three derived columns and one removed column.

In [9]:
print("Original shape:", original_df.shape)
print("Processed shape:", df.shape)
print("Remaining exact duplicate rows:", df.duplicated().sum())
display(df.isna().sum().to_frame("Remaining missing values"))
display(df.head())

Original shape: (2073, 25)
Processed shape: (2073, 27)
Remaining exact duplicate rows: 0


,Remaining missing values
X,0
Y,0
CCN,0
REPORT_DAT,0
START_DATE,1
END_DATE,178
BLOCK,0
OFFENSE,0
METHOD,0
SHIFT,0


,X,Y,CCN,REPORT_DAT,START_DATE,END_DATE,BLOCK,OFFENSE,METHOD,SHIFT,WARD,ANC,DISTRICT,PSA,NEIGHBORHOOD_CLUSTER,BLOCK_GROUP,CENSUS_TRACT,VOTING_PRECINCT,BID,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID,Recorded_Interval_Hours,Interval_Normalized,Interval_Bin
0,397953.59,140415.23,26137330,2026-09-26 06:54:25+00:00,2026-09-25 22:36:00+00:00,2026-09-25 23:34:00+00:00,3301 - 3399 BLOCK OF GEORGIA AVENUE NW,THEFT/OTHER,OTHERS,MIDNIGHT,1,1E,4,409,Cluster 2,003100 2,3100,Precinct 38,Unknown,397953.59,140415.23,38.931616,-77.023602,1042684332,0.966667,0.000182,Short interval
1,398099.12,138058.27,26137481,2026-09-26 05:51:18+00:00,2026-09-26 04:49:00+00:00,2026-09-26 05:45:00+00:00,1500 - 1599 BLOCK OF 7TH STREET NW,THEFT/OTHER,OTHERS,MIDNIGHT,2,2G,3,307,Cluster 7,004901 2,4901,Precinct 18,Unknown,398099.12,138058.27,38.910384,-77.021917,1042684333,0.933333,0.000176,Short interval
2,397525.88,138018.68,26138263,2026-09-27 23:11:57+00:00,2026-09-25 00:15:00+00:00,2026-09-25 02:00:00+00:00,1200 - 1299 BLOCK OF RHODE ISLAND AVENUE NW,THEFT F/AUTO,OTHERS,EVENING,2,2F,3,307,Cluster 7,005001 1,5001,Precinct 16,Unknown,397525.88,138018.68,38.910026,-77.028526,1042684334,1.750000,0.000330,Short interval
3,398682.30,134716.49,26139697,2026-09-30 15:06:32+00:00,2026-09-29 12:00:00+00:00,2026-09-29 20:00:00+00:00,700 - 899 BLOCK OF 3RD STREET SW,THEFT/OTHER,OTHERS,DAY,6,6D,1,103,Cluster 9,010500 2,10500,Precinct 128,SOUTHWEST,398682.30,134716.49,38.880281,-77.015187,1042684335,8.000000,0.001508,Short interval
4,397209.47,139586.90,26142075,2026-10-05 00:01:20+00:00,2026-10-04 23:37:00+00:00,2026-10-05 00:00:00+00:00,2600 - 2698 BLOCK OF 14TH STREET NW,THEFT/OTHER,OTHERS,EVENING,1,1A,3,304,Cluster 2,003600 2,3600,Precinct 36,Unknown,397209.47,139586.90,38.924152,-77.032181,1042684721,0.383333,0.000072,Short interval


### Phase 1 conclusion
The dataset contains 2,073 records and initially 25 columns. There were no exact duplicate rows. Missing categories were labelled Unknown and the entirely empty OCTO_RECORD_ID column was removed. Date conversion introduced no additional invalid timestamps. No reversed start/end pairs or coordinates outside global bounds were found.

There are 1,895 available recorded intervals and 178 missing intervals. The longest interval is 5,304 hours. These long windows should be reviewed in Phase 2; they have not been automatically removed. Normalization and equal-width binning were demonstrated while preserving original measurements.

**Next phase:** Statistical analysis, distribution shape, IQR and boxplot outlier investigation, and appropriate proximity measures.

**Sources:** Supplied Crime_Incidents_in_the_Last_30_Days (1).csv; supplied FDS Mini Competition Guidelines and evaluation rubric.